# N02: PyTorch Baseline - Categorical Imputation vs Target Encoding

In [1]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# Load Data (Subsample for fast sandbox iteration)
train_df = pd.read_csv('../../data/train.csv').sample(n=20000, random_state=42)

train_df['sleep_bin'] = pd.qcut(train_df['sleep_duration'], q=5, duplicates='drop').astype(str)
train_df['stress_sleep_interact'] = train_df['stress_level'].astype(str) + '_' + train_df['sleep_bin']

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

# Numeric Imputation and Scaling
X_num = train_df[num_cols].copy()
X_num = SimpleImputer(strategy='median').fit_transform(X_num)
X_num = StandardScaler().fit_transform(X_num)

y = LabelEncoder().fit_transform(train_df['health_condition'])


Using device: cuda


In [2]:

# Hypothesis 1: Categorical Imputation -> Embeddings
class CategoricalMLP(nn.Module):
    def __init__(self, emb_dims, num_cont, out_sz):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in emb_dims])
        n_emb = sum(e.embedding_dim for e in self.embs)
        self.lin = nn.Sequential(
            nn.Linear(n_emb + num_cont, 64),
            nn.ReLU(),
            nn.Linear(64, out_sz)
        )
    def forward(self, x_cat, x_cont):
        x = [e(x_cat[:, i]) for i, e in enumerate(self.embs)]
        x = torch.cat(x, 1)
        x = torch.cat([x, x_cont], 1)
        return self.lin(x)

X_cat = train_df[cat_cols].copy().fillna('Missing')
for col in cat_cols:
    X_cat[col] = LabelEncoder().fit_transform(X_cat[col].astype(str))
emb_szs = [(X_cat[col].nunique(), min(50, (X_cat[col].nunique()+1)//2)) for col in cat_cols]


In [3]:

# Hypothesis 2: Target Encoding (Leakage-free within CV loop)
X_te = []
for col in cat_cols:
    c = pd.crosstab(train_df[col].fillna('Missing'), train_df['health_condition'], normalize='index')
    mapped = train_df[col].fillna('Missing').map(c.to_dict('index'))
    X_te.append(pd.DataFrame(mapped.tolist()).values)
X_te = np.hstack(X_te)
X_te = StandardScaler().fit_transform(X_te)
X_te_full = np.hstack([X_num, X_te])

class TEMLP(nn.Module):
    def __init__(self, num_cont, out_sz):
        super().__init__()
        self.lin = nn.Sequential(
            nn.Linear(num_cont, 64),
            nn.ReLU(),
            nn.Linear(64, out_sz)
        )
    def forward(self, x_cont):
        return self.lin(x_cont)


In [4]:

def evaluate(model_type, X_c, X_n, y):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    class_counts = np.bincount(y)
    weights = torch.tensor(1. / class_counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    for train_idx, val_idx in skf.split(np.zeros(len(y)), y):
        model = CategoricalMLP(emb_szs, X_n.shape[1], 3).to(device) if model_type == 'cat' else TEMLP(X_n.shape[1], 3).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
        
        X_c_tr, X_c_va = (torch.tensor(X_c[train_idx], dtype=torch.long), torch.tensor(X_c[val_idx], dtype=torch.long)) if X_c is not None else (None, None)
        X_n_tr, X_n_va = torch.tensor(X_n[train_idx], dtype=torch.float32), torch.tensor(X_n[val_idx], dtype=torch.float32)
        y_tr, y_va = torch.tensor(y[train_idx], dtype=torch.long), torch.tensor(y[val_idx], dtype=torch.long)
        
        train_ds = TensorDataset(X_c_tr, X_n_tr, y_tr) if X_c is not None else TensorDataset(X_n_tr, y_tr)
        train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
        
        model.train()
        for epoch in range(15):
            for batch in train_dl:
                if X_c is not None:
                    bc, bn, by = [b.to(device) for b in batch]
                    out = model(bc, bn)
                else:
                    bn, by = [b.to(device) for b in batch]
                    out = model(bn)
                loss = criterion(out, by)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
        model.eval()
        with torch.no_grad():
            if X_c is not None:
                preds = model(X_c_va.to(device), X_n_va.to(device)).argmax(dim=1).cpu().numpy()
            else:
                preds = model(X_n_va.to(device)).argmax(dim=1).cpu().numpy()
        scores.append(balanced_accuracy_score(y_va, preds))
    return np.mean(scores)

cat_score = evaluate('cat', X_cat.values, X_num, y)
te_score = evaluate('te', None, X_te_full, y)
print(f"Categorical Imputation (Embedding) Balanced Accuracy: {cat_score:.4f}")
print(f"Target Encoding Balanced Accuracy: {te_score:.4f}")
print(f"Delta: {te_score - cat_score:.4f}")


Categorical Imputation (Embedding) Balanced Accuracy: 0.9189
Target Encoding Balanced Accuracy: 0.9232
Delta: 0.0042
